# Clinical Severity Scoring — CURB65 Boundary Analysis

**Precise clinical severity scoring across CURB65 boundary conditions.**

> This notebook is generated by `_generate_severity_scoring_notebook.py`. Do not edit manually.

### Overview

CURB65 is the standard severity scoring system for community-acquired pneumonia (CAP).
Each variable contributes 1 point when its threshold is crossed:

- **C**onfusion (AMT score)
- **U**rea (mmol/L)
- **R**espiratory rate (breaths/min)
- **B**lood pressure (systolic and diastolic)
- Age >= **65** years

The total score (0-5) determines severity tier and treatment pathway.

### Clinical Importance

Boundary precision is safety-critical. A urea of 7.0 vs 7.1 mmol/L changes the score,
which can change the treatment pathway from oral antibiotics at home to IV therapy in hospital.
This notebook runs **15 Group A cases** through the full pipeline to verify every boundary
is handled correctly:

- **12 single-variable boundary tests** — each CURB65 variable tested at its exact threshold
- **3 composite severity cases** — low (CURB65=0), moderate (CURB65=2), high (CURB65=4)

### Evidence Base

Scoring thresholds are implemented as deterministic clinical logic in
`cap_agent.agent.clinical_logic` implements these thresholds with exact boundary semantics.

### Quick Start
1. Set runtime to **GPU -> A100** (Runtime -> Change runtime type)
2. Add **HF_TOKEN** to Colab Secrets (key icon in left sidebar)
3. *(Optional)* Add **GITHUB_TOKEN** for private repo install
4. **Run All** (Runtime -> Run all)


> **Disclaimer:** This notebook demonstrates a research prototype. All clinical outputs
> (severity scores, antibiotic recommendations, contradiction alerts, CXR analysis)
> are **AI-generated by MedGemma 1.5 4B** and have been validated only on synthetic data.
> This system is **not a medical device**, is not approved for clinical use, and must not
> be used for patient care decisions. See [DISCLAIMER.md](../DISCLAIMER.md) for full terms.


In [ ]:
# Cell 1: Install package + dependencies
import os

# Detect Colab vs local
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install from GitHub (requires GITHUB_TOKEN in Colab Secrets)
    try:
        github_token = userdata.get("GITHUB_TOKEN")
        repo_url = f"git+https://{github_token}@github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"
    except Exception:
        repo_url = "git+https://github.com/HP-00/MedGemma-Pneumonia-Agent.git@main"

    %pip install --quiet {repo_url}
    %pip install --quiet plotly>=5.0.0 pandas>=2.0.0 nest-asyncio
else:
    print("Local environment detected. Ensure: pip install -e '.[dev,benchmark]'")

import nest_asyncio
nest_asyncio.apply()

print("Setup complete")


## Authentication


In [ ]:
import os

# HuggingFace token for gated model access
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set! Add it to Colab Secrets or environment."

print("Authentication complete")


## Load Model & Build Graph


In [ ]:
import time
from cap_agent.models.medgemma import get_model_and_processor
from cap_agent.agent.graph import build_cap_agent_graph

# Load MedGemma (lazy singleton - only loads once, includes warm-up)
model, processor = get_model_and_processor()
print(f"Model loaded on {model.device}")

# Build the 8-node LangGraph pipeline
graph = build_cap_agent_graph()
print("Graph compiled: 8 nodes, conditional routing at contradiction check")


## CURB65 Scoring Rules

Each variable scores 0 or 1. The total (0-5) determines severity tier.

| Variable | Criterion | Score |
|----------|-----------|-------|
| **C**onfusion | AMT <= 8 | 1 |
| **U**rea | > 7 mmol/L (strictly greater) | 1 |
| **R**espiratory rate | >= 30/min | 1 |
| **B**lood pressure | SBP < 90 **OR** DBP <= 60 | 1 |
| Age >= **65** | >= 65 years | 1 |

### Severity Tiers

| CURB65 Score | Severity | Management |
|-------------|----------|------------|
| 0-1 | Low | Consider community treatment |
| 2 | Moderate | Hospital assessment / short stay |
| 3-5 | High | Hospital admission, consider ICU if 4-5 |

### Boundary Precision

Note the mixed comparison operators — these are clinically defined and must be implemented exactly:
- Urea uses **strictly greater than** (>), not >=. So urea=7.0 scores 0 but urea=7.1 scores 1.
- SBP uses **strictly less than** (<). So SBP=90 scores 0 but SBP=89 scores 1.
- DBP uses **less than or equal** (<=). So DBP=61 scores 0 but DBP=60 scores 1.
- RR uses **greater than or equal** (>=). So RR=29 scores 0 but RR=30 scores 1.
- Age uses **greater than or equal** (>=). So age=64 scores 0 but age=65 scores 1.
- AMT uses **less than or equal** (<=). So AMT=9 scores 0 but AMT=8 scores 1.


## Load Group A Benchmark Cases

15 cases from the benchmark case registry, split into:
- **12 boundary cases** (`CURB65-*`): Each tests a single CURB65 variable at its exact threshold
- **3 composite cases** (`CURB65-COMPOSITE-*`): Low, moderate, and high severity with multiple variables active


In [ ]:
from benchmark_data.cases.registry import get_track2_cases

all_track2 = get_track2_cases()

# Group A: CURB65 boundary cases
group_a = [c for c in all_track2 if c["case_id"].startswith("CURB65-")]

print(f"Loaded {len(group_a)} severity scoring cases")

boundary_cases = [c for c in group_a if not c["case_id"].startswith("CURB65-COMPOSITE-")]
composite_cases = [c for c in group_a if c["case_id"].startswith("CURB65-COMPOSITE-")]

print(f"  Boundary tests: {len(boundary_cases)}")
print(f"  Composite cases: {len(composite_cases)}")

# Show case IDs
print("\nBoundary cases:")
for c in boundary_cases:
    gt = c["ground_truth"]
    print(f"  {c['case_id']}: expected CURB65={gt['curb65']} ({gt['severity_tier']})")

print("\nComposite cases:")
for c in composite_cases:
    gt = c["ground_truth"]
    print(f"  {c['case_id']}: expected CURB65={gt['curb65']} ({gt['severity_tier']})")


In [ ]:
# Clinical output renderer — formats structured_output as a readable document
def render_clinical_output(result, scenario_title="Pipeline Output"):
    """Render structured_output as a formatted clinical document using ANSI codes."""
    ESC = chr(27)
    BOLD = f"{ESC}[1m"
    RED = f"{ESC}[91m"
    YEL = f"{ESC}[93m"
    GRN = f"{ESC}[92m"
    CYN = f"{ESC}[96m"
    RST = f"{ESC}[0m"

    so = result.get("structured_output", {})
    sections = so.get("sections", {})

    print(f"\n{BOLD}{CYN}{'=' * 70}")
    print(f"  CLINICAL DECISION SUPPORT — {scenario_title}")
    print(f"{'=' * 70}{RST}")
    print(f"{YEL}AI-generated observations for clinician review — not a substitute for clinical judgement.{RST}\n")

    # 1. PATIENT
    s1 = sections.get("1_patient", {})
    demo = s1.get("demographics", {})
    print(f"{BOLD}{CYN}1. PATIENT{RST}")
    print(f"  {demo.get('age', '?')}yo {demo.get('sex', '?')}")
    allergy_list = demo.get("allergies", [])
    if allergy_list:
        for a in allergy_list:
            if isinstance(a, dict):
                print(f"  {RED}Allergy: {a.get('drug', '?')} — {a.get('reaction_type', '?')} ({a.get('severity', '?')}){RST}")
            else:
                print(f"  {RED}Allergy: {a}{RST}")
    else:
        print(f"  {GRN}No known drug allergies{RST}")
    print()

    # 2. SEVERITY
    s2 = sections.get("2_severity", {})
    curb = s2.get("curb65", {})
    sev = curb.get("severity_tier", "?")
    if sev == "high":
        sev_color = RED
    elif sev == "moderate":
        sev_color = YEL
    else:
        sev_color = GRN
    score = curb.get("curb65")
    score_str = f"CURB65={score}" if score is not None else f"CRB65={curb.get('crb65', '?')} (urea unavailable)"
    print(f"{BOLD}{CYN}2. SEVERITY{RST}")
    print(f"  {score_str}  {sev_color}{BOLD}{sev.upper()}{RST}")
    print(f"  C={curb.get('c','?')} U={curb.get('u','?')} R={curb.get('r','?')} B={curb.get('b','?')} 65={curb.get('age_65','?')}")
    missing = curb.get("missing_variables", [])
    if missing:
        print(f"  {YEL}Missing: {', '.join(missing)}{RST}")
    poc = s2.get("place_of_care", {})
    if poc:
        print(f"  Place of care: {poc.get('recommendation', '?')}")
    print()

    # 3. TREATMENT
    s6 = sections.get("6_treatment_pathway", {})
    abx = s6.get("antibiotic", {})
    print(f"{BOLD}{CYN}3. TREATMENT{RST}")
    print(f"  First-line: {abx.get('first_line', 'N/A')}")
    print(f"  Dose/route: {abx.get('dose_route', 'N/A')}")
    if abx.get("allergy_adjustment"):
        print(f"  {YEL}Allergy adj: {abx['allergy_adjustment']}{RST}")
    if abx.get("atypical_cover"):
        print(f"  Atypical: {abx['atypical_cover']}")
    print()

    # PROVENANCE
    prov = so.get("provenance", {})
    if prov:
        print(f"{BOLD}{CYN}PROVENANCE{RST}")
        tools = prov.get("extraction_tools", {})
        sources = prov.get("data_sources", {})
        for tool_name, pipeline in tools.items():
            src = sources.get(tool_name, [])
            print(f"  {tool_name}: {pipeline} ({', '.join(src) if src else 'N/A'})")

    print(f"\n{CYN}{'=' * 70}{RST}\n")

print("render_clinical_output() defined")


## Run All 15 Cases Through the Pipeline

Each case runs the full 8-node LangGraph pipeline. Since these are synthetic cases
without CXR images or FHIR bundles, extraction uses mock passthrough (0 GPU calls for
EHR/Lab/CXR). The clinical logic nodes (severity scoring, treatment selection, monitoring)
are fully deterministic and do not depend on extraction quality.


In [ ]:
from cap_agent.agent.state import build_initial_state

all_results = []

for i, case in enumerate(group_a):
    case_id = case["case_id"]
    print(f"[{i+1}/{len(group_a)}] Running {case_id}...", end=" ")

    initial_state = build_initial_state(case)

    start = time.time()
    result = None
    for chunk in graph.stream(initial_state, stream_mode="values"):
        result = chunk
    elapsed = time.time() - start

    curb = result.get("curb65_score", {})
    gt = case["ground_truth"]
    expected = gt["curb65"]
    actual = curb.get("curb65")
    match = actual == expected
    status = "CORRECT" if match else "WRONG"

    all_results.append({
        "case_id": case_id,
        "case": case,
        "result": result,
        "elapsed": elapsed,
        "match": match,
    })

    print(f"CURB65={actual} (expected {expected}) — {status} [{elapsed:.1f}s]")

passed = sum(1 for r in all_results if r["match"])
print(f"\n{passed}/{len(all_results)} cases scored correctly")


## Boundary Analysis: Confusion (C)

**Threshold:** AMT <= 8 scores C=1 (confused)

| Case | AMT Score | Expected C | Expected CURB65 |
|------|-----------|-----------|-----------------|
| CURB65-AMT-BELOW | 9 | 0 | 0 |
| CURB65-AMT-ABOVE | 8 | 1 | 1 |

AMT (Abbreviated Mental Test) scores 0-10. A score of 8 or below indicates confusion.
Note this uses **less than or equal** — AMT=8 means confused.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; RST = f"{ESC}[0m"

print(f"{BOLD}Confusion (C) — AMT <= 8{RST}\n")

var_cases = [r for r in all_results if r["case_id"].startswith("CURB65-AMT-")]
for r in var_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    amt = r["case"]["clinical_exam"]["confusion_assessment"]["amt_score"]
    print(f"  {r['case_id']}: AMT={amt} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")


## Boundary Analysis: Urea (U)

**Threshold:** Urea > 7.0 mmol/L scores U=1 (strictly greater than)

| Case | Urea (mmol/L) | Expected U | Expected CURB65 |
|------|--------------|-----------|-----------------|
| CURB65-U-BELOW | 7.0 | 0 | 0 |
| CURB65-U-ABOVE | 7.1 | 1 | 1 |

This is the most commonly misimplemented boundary. The threshold is **strictly greater than**
7.0, meaning urea=7.0 does NOT score a point. This matches the original CURB65 validation
study.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; RST = f"{ESC}[0m"

print(f"{BOLD}Urea (U) — strictly > 7.0 mmol/L{RST}\n")

var_cases = [r for r in all_results if r["case_id"].startswith("CURB65-U-")]
for r in var_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    urea = r["case"]["lab_results"]["urea"]["value"]
    print(f"  {r['case_id']}: urea={urea} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")


## Boundary Analysis: Respiratory Rate (R)

**Threshold:** RR >= 30/min scores R=1

| Case | RR (/min) | Expected R | Expected CURB65 |
|------|-----------|-----------|-----------------|
| CURB65-R-BELOW | 29 | 0 | 0 |
| CURB65-R-ABOVE | 30 | 1 | 1 |

Respiratory rate uses **greater than or equal** — RR=30 scores a point.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; RST = f"{ESC}[0m"

print(f"{BOLD}Respiratory Rate (R) — >= 30/min{RST}\n")

var_cases = [r for r in all_results if r["case_id"].startswith("CURB65-R-")]
for r in var_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    rr = r["case"]["clinical_exam"]["observations"]["respiratory_rate"]
    print(f"  {r['case_id']}: RR={rr} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")


## Boundary Analysis: Blood Pressure (B)

**Threshold:** SBP < 90 **OR** DBP <= 60 scores B=1

Two sub-thresholds with different comparison operators:

| Case | SBP | DBP | Expected B | Expected CURB65 |
|------|-----|-----|-----------|-----------------|
| CURB65-SBP-BELOW | 90 | 70 | 0 | 0 |
| CURB65-SBP-ABOVE | 89 | 70 | 1 | 1 |
| CURB65-DBP-BELOW | 120 | 61 | 0 | 0 |
| CURB65-DBP-ABOVE | 120 | 60 | 1 | 1 |

- SBP uses **strictly less than** (<) — SBP=90 does NOT score a point
- DBP uses **less than or equal** (<=) — DBP=60 DOES score a point
- Either condition being true is sufficient for B=1


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; RST = f"{ESC}[0m"

print(f"{BOLD}Blood Pressure (B) — SBP < 90 OR DBP <= 60{RST}\n")

# SBP boundary
print("  SBP boundary (strictly < 90):")
sbp_cases = [r for r in all_results if r["case_id"].startswith("CURB65-SBP-")]
for r in sbp_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    sbp = r["case"]["clinical_exam"]["observations"]["systolic_bp"]
    print(f"    {r['case_id']}: SBP={sbp} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")

# DBP boundary
print("\n  DBP boundary (<= 60):")
dbp_cases = [r for r in all_results if r["case_id"].startswith("CURB65-DBP-")]
for r in dbp_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    dbp = r["case"]["clinical_exam"]["observations"]["diastolic_bp"]
    print(f"    {r['case_id']}: DBP={dbp} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")


## Boundary Analysis: Age (65)

**Threshold:** Age >= 65 scores Age=1

| Case | Age | Expected 65 | Expected CURB65 |
|------|-----|-----------|-----------------|
| CURB65-AGE-BELOW | 64 | 0 | 0 |
| CURB65-AGE-ABOVE | 65 | 1 | 1 |

Age uses **greater than or equal** — age=65 scores a point.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; RST = f"{ESC}[0m"

print(f"{BOLD}Age (65) — >= 65 years{RST}\n")

var_cases = [r for r in all_results if r["case_id"].startswith("CURB65-AGE-")]
for r in var_cases:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    match = actual_curb65 == expected_curb65
    color = GRN if match else RED
    status = "CORRECT" if match else "WRONG"
    age = r["case"]["demographics"]["age"]
    print(f"  {r['case_id']}: age={age} -> CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')} — {color}{status}{RST}")


## Composite Severity Cases

Three cases testing the full CURB65 score and its mapping to severity tiers:

| Case | Active Variables | CURB65 | Severity |
|------|-----------------|--------|----------|
| CURB65-COMPOSITE-LOW | None (all baseline) | 0 | Low |
| CURB65-COMPOSITE-MOD | Age=65, Urea=7.1 | 2 | Moderate |
| CURB65-COMPOSITE-HIGH | Age=65, Urea=7.1, RR=30, AMT=8 | 4 | High |

These cases verify that the pipeline correctly maps the total score to the severity tier
and selects the appropriate treatment pathway.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; YEL = f"{ESC}[93m"
CYN = f"{ESC}[96m"; RST = f"{ESC}[0m"

print(f"{BOLD}{CYN}Composite Severity Cases{RST}\n")

comp_results = [r for r in all_results if r["case_id"].startswith("CURB65-COMPOSITE-")]
for r in comp_results:
    result = r["result"]
    curb = result.get("curb65_score", {})
    gt = r["case"]["ground_truth"]
    expected_curb65 = gt["curb65"]
    expected_tier = gt["severity_tier"]
    actual_curb65 = curb.get("curb65")
    actual_tier = curb.get("severity_tier")

    score_match = actual_curb65 == expected_curb65
    tier_match = actual_tier == expected_tier
    all_match = score_match and tier_match

    color = GRN if all_match else RED
    status = "CORRECT" if all_match else "WRONG"

    if actual_tier == "high":
        tier_color = RED
    elif actual_tier == "moderate":
        tier_color = YEL
    else:
        tier_color = GRN

    print(f"  {BOLD}{r['case_id']}{RST}")
    print(f"    CURB65={actual_curb65} (expected {expected_curb65}) "
          f"C={curb.get('c')} U={curb.get('u')} R={curb.get('r')} B={curb.get('b')} "
          f"65={curb.get('age_65')}")
    print(f"    Severity: {tier_color}{actual_tier}{RST} (expected {expected_tier}) — {color}{status}{RST}")

    # Treatment pathway
    abx = result.get("antibiotic_recommendation", {})
    print(f"    First-line: {abx.get('first_line', 'N/A')}")
    print(f"    Dose/route: {abx.get('dose_route', 'N/A')}")

    poc = result.get("structured_output", {}).get("sections", {}).get("2_severity", {}).get("place_of_care", {})
    if poc:
        print(f"    Place of care: {poc.get('recommendation', 'N/A')}")
    print()

# Render full clinical output for each composite case
for r in comp_results:
    render_clinical_output(r["result"], r["case_id"])


## Severity to Treatment Pathway Mapping

The CURB65 severity tier directly determines the antibiotic regimen, route, and care setting:

| Severity | CURB65 | Antibiotic | Route | Setting |
|----------|--------|------------|-------|---------|
| Low | 0-1 | Amoxicillin 500mg TDS | PO (oral) | Community |
| Moderate | 2 | Amoxicillin +/- clarithromycin | PO (oral) | Hospital |
| High | 3-5 | Co-amoxiclav IV + clarithromycin | IV | Hospital / ICU |

This mapping follows severity-stratified treatment recommendations. The pipeline implements this as a
deterministic rule engine in `select_antibiotic()`, ensuring every boundary change in
CURB65 propagates correctly to the treatment recommendation.

Note that additional modifiers (penicillin allergy, renal impairment, pregnancy, travel
history) can adjust the specific antibiotic choice but do not change the severity-driven
treatment intensity.


## Summary: Boundary Accuracy

Heatmap showing correctness of each CURB65 variable boundary test.


In [ ]:
import plotly.graph_objects as go

# Build boundary accuracy matrix
variables = ["AMT (C)", "Urea (U)", "RR (R)", "SBP (B)", "DBP (B)", "Age (65)"]
sides = ["Below threshold", "At/above threshold"]

prefixes = ["CURB65-AMT-", "CURB65-U-", "CURB65-R-", "CURB65-SBP-", "CURB65-DBP-", "CURB65-AGE-"]
suffixes = ["BELOW", "ABOVE"]

z_values = []
annotations = []

for i, prefix in enumerate(prefixes):
    row = []
    for j, suffix in enumerate(suffixes):
        case_id = f"{prefix}{suffix}"
        r = next((r for r in all_results if r["case_id"] == case_id), None)
        if r is not None:
            correct = 1 if r["match"] else 0
            row.append(correct)
            label = "CORRECT" if r["match"] else "WRONG"
            curb = r["result"].get("curb65_score", {})
            annotations.append(dict(
                x=j, y=i,
                text=f"{label}<br>CURB65={curb.get('curb65')}",
                showarrow=False,
                font=dict(color="white" if correct == 1 else "black", size=11),
            ))
        else:
            row.append(-1)
    z_values.append(row)

fig = go.Figure(data=go.Heatmap(
    z=z_values,
    x=sides,
    y=variables,
    colorscale=[[0, "#EF5350"], [1, "#66BB6A"]],
    showscale=False,
    zmin=0, zmax=1,
))
fig.update_layout(
    title="CURB65 Boundary Accuracy (12 tests)",
    annotations=annotations,
    width=500, height=400,
    yaxis=dict(autorange="reversed"),
)
fig.show()

# Text summary
correct_count = sum(1 for r in all_results if not r["case_id"].startswith("CURB65-COMPOSITE-") and r["match"])
total_boundary = sum(1 for r in all_results if not r["case_id"].startswith("CURB65-COMPOSITE-"))
print(f"\nBoundary accuracy: {correct_count}/{total_boundary}")


## Summary: Severity Distribution

Bar chart showing the distribution of severity tiers across all 15 cases.


In [ ]:
import plotly.graph_objects as go

# Count severity tiers
tier_counts = {"low": 0, "moderate": 0, "high": 0}
for r in all_results:
    curb = r["result"].get("curb65_score", {})
    tier = curb.get("severity_tier", "unknown")
    if tier in tier_counts:
        tier_counts[tier] += 1

tiers = ["low", "moderate", "high"]
counts = [tier_counts[t] for t in tiers]
colors = ["#66BB6A", "#FFA726", "#EF5350"]

fig = go.Figure(data=go.Bar(
    x=tiers,
    y=counts,
    marker_color=colors,
    text=counts,
    textposition="auto",
))
fig.update_layout(
    title="Severity Tier Distribution (15 cases)",
    xaxis_title="Severity Tier",
    yaxis_title="Number of Cases",
    width=500, height=350,
    yaxis=dict(dtick=1),
)
fig.show()

print(f"\nSeverity distribution: low={tier_counts['low']}, "
      f"moderate={tier_counts['moderate']}, high={tier_counts['high']}")


## Verification Checklist

All 15 cases: expected vs actual CURB65 score and severity tier.


In [ ]:
ESC = chr(27)
BOLD = f"{ESC}[1m"; GRN = f"{ESC}[92m"; RED = f"{ESC}[91m"; YEL = f"{ESC}[93m"
CYN = f"{ESC}[96m"; RST = f"{ESC}[0m"

print(f"\n{BOLD}{CYN}{'=' * 70}")
print(f"  SEVERITY SCORING VERIFICATION — ALL 15 CASES")
print(f"{'=' * 70}{RST}\n")

header = f"  {'Case ID':<28} {'Expected':>10} {'Actual':>10} {'Tier':>10} {'Status':>10}"
print(header)
print(f"  {'─' * 68}")

total_correct = 0
total_cases = len(all_results)

for r in all_results:
    curb = r["result"].get("curb65_score", {})
    gt = r["case"]["ground_truth"]

    expected_curb65 = gt["curb65"]
    actual_curb65 = curb.get("curb65")
    expected_tier = gt["severity_tier"]
    actual_tier = curb.get("severity_tier")

    score_ok = actual_curb65 == expected_curb65
    tier_ok = actual_tier == expected_tier
    all_ok = score_ok and tier_ok

    if all_ok:
        total_correct += 1

    color = GRN if all_ok else RED
    status = "CORRECT" if all_ok else "WRONG"

    print(f"  {r['case_id']:<28} {str(expected_curb65):>10} {str(actual_curb65):>10} "
          f"{actual_tier or '?':>10} {color}{status}{RST:>10}")

print(f"\n{BOLD}Results: {total_correct}/{total_cases} cases scored correctly{RST}")

if total_correct == total_cases:
    print(f"\n{GRN}{BOLD}All severity scoring checks passed! "
          f"Every CURB65 boundary is handled correctly.{RST}")
else:
    failed = total_cases - total_correct
    print(f"\n{RED}{BOLD}{failed} case(s) failed — review boundary implementation.{RST}")
